# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant Dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets in the dataset (print their @id and name)
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id={rs.id}, name={rs.name}")

# Print fields for each record set
for rs in record_sets:
    print(f"\nFields in record set '@id={rs.id}', name='{rs.name}':")
    for field in rs.fields:
        field_details = f"  - @id={field.id}, name={field.name}"
        if hasattr(field, 'data_type'):
            field_details += f", type={field.data_type}"
        print(field_details)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we select the main protein abundance record set
# Typically the largest record set contains the main data (bioinformatics/proteomics record set)
# Record set ids are provided in the previous step. Adjust if you want to analyze a different record set.
main_record_set_id = None
for rs in record_sets:
    # Choose the record set with the most fields as a proxy for the main table
    if main_record_set_id is None or len(rs.fields) > len(main_rs.fields):
        main_rs = rs
        main_record_set_id = rs.id
assert main_record_set_id is not None
print(f"Selected main record set: @id={main_record_set_id} (name={main_rs.name})")

records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print("Columns in main DataFrame:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll:
- Select a numeric field (e.g., protein abundance, MW, or coverage_percent) using the field `@id`.
- Filter and process the DataFrame, showing best practice usage of Croissant field identifiers.

In [ ]:
# Find numeric fields: We'll look for typical proteomics numeric columns by name or @id
import numpy as np

# Find candidate numeric fields (@id/name known from previous cell)
numeric_candidates = []
for field in main_rs.fields:
    # Try to pick likely numeric fields
    col_id = field.id
    if (hasattr(field, 'data_type') and field.data_type in ['http://schema.org/Number', 'http://schema.org/Float', 'http://schema.org/Integer']):
        numeric_candidates.append(col_id)
    elif any(key in col_id.lower() for key in ['abundance', 'mw', 'coverage', 'count']):
        numeric_candidates.append(col_id)
        
print("Numeric field candidates for EDA:", numeric_candidates)

# For demonstration, pick the first numeric field
numeric_field_id = numeric_candidates[0] if numeric_candidates else df.select_dtypes(include=np.number).columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Remove non-numeric rows if any (e.g. missing values)
df_numeric = df.copy()
df_numeric = df_numeric[pd.to_numeric(df_numeric[numeric_field_id], errors='coerce').notnull()]
df_numeric[numeric_field_id] = df_numeric[numeric_field_id].astype(float)

# Filter: only consider rows with value > threshold
threshold = df_numeric[numeric_field_id].quantile(0.9)  # 90th percentile as example
filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Add a normalized column (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - df_numeric[numeric_field_id].mean()) / df_numeric[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try group by a categorical field: use a field with string values
group_field_id = None
for field in main_rs.fields:
    col_id = field.id
    if col_id != numeric_field_id and df_numeric[col_id].dtype == object:
        group_field_id = col_id
        break

if group_field_id and group_field_id in df_numeric:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Examples: Histogram, boxplot, or scatterplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (all records)
plt.figure(figsize=(8, 4))
sns.histplot(df_numeric[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouped data is available, show a barplot
if 'grouped_df' in locals() and group_field_id:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df.head(10))
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (top 10 groups)")
    plt.xticks(rotation=75)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and analyze the Mass Spectrometry dataset using the `mlcroissant` library by referring to record sets and fields exclusively by their `@id`. This approach ensures reproducibility and precise referencing in complex, schema-driven data packages like FAIR^2. Feel free to extend this analysis with additional fields or more advanced visualizations.